In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

from core.problems.tsp.graph_solution import TSPGraphSolution

reward = torch.tensor([[1, 2, 3], [3, 2, 1], [0, 10, 0]], dtype=torch.float)
log_p = torch.randn_like(reward)

b = reward.mean(dim=1, keepdim=True)
k_star = torch.argmax(reward, dim=1, keepdim=True)

best_rewards = torch.gather(reward, 1, k_star)
best_log_p = torch.gather(log_p, 1, k_star)

In [3]:
from core.problems.tsp import TSP, TSPSampler
from core.problems.tsp.solve import random_permutation

tsp_instance = TSPSampler(TSP(num_locations=10), seed=42)(4)
tsp_solution = random_permutation(tsp_instance)

In [4]:
print(tsp_solution.tour)

tensor([[4, 6, 0, 5, 8, 3, 2, 1, 7, 9],
        [3, 7, 8, 0, 6, 2, 9, 4, 1, 5],
        [4, 2, 9, 1, 0, 7, 8, 5, 3, 6],
        [8, 0, 1, 6, 7, 4, 3, 2, 5, 9]])


In [5]:
solution = TSPGraphSolution.create_from_sequence(tsp_instance, tsp_solution.tour)

In [6]:
print(solution.next_nodes)

tensor([[5, 7, 1, 2, 6, 8, 0, 9, 3, 4],
        [6, 5, 9, 7, 1, 3, 2, 8, 0, 4],
        [7, 0, 9, 6, 2, 3, 4, 8, 5, 1],
        [1, 6, 5, 2, 3, 9, 7, 4, 0, 8]])


In [7]:
import pickle

with open("../data/synthetic/cvrp/vrp100_test_seed1234.pkl", "rb") as f:
    cvrp100_test_dataset = pickle.load(f)

In [8]:
depot, locations, demand, capacity = cvrp100_test_dataset[0]

print(depot)
print(locations)
print(demand)
print(capacity)

[0.1915194503788923, 0.6221087710398319]
[[0.5542693865183056, 0.1809782379192011], [0.37612515633291255, 0.5188554893458212], [0.05188428813232193, 0.7671967617203561], [0.04856034174547064, 0.8196928268696034], [0.5749099573002305, 0.8349017687417157], [0.8452234825824089, 0.1786910149407145], [0.9165613605771327, 0.9391639577090866], [0.06335356542623072, 0.8306785088648821], [0.708990050257321, 0.25709061845325953], [0.3247405554511438, 0.5733085690335363], [0.8432614130569878, 0.29342090222703354], [0.010939144115935329, 0.4625811201110085], [0.3587623438232923, 0.194003726127849], [0.7212015125029465, 0.7955572179769141], [0.05224471966802857, 0.5844070535639075], [0.7063008963717297, 0.2374767808471393], [0.030432550498018185, 0.9948726146076882], [0.15485826266303093, 0.45972601008249225], [0.0018524517245349648, 0.4513273215226944], [0.9878568671667981, 0.25486374091014385], [0.24768139409327772, 0.7757472559661192], [0.9364897590754794, 0.98855370646227], [0.7879112404331364,

In [10]:
import torch

path = "../experiments/pretrained/nds/cvrp_100/checkpoint-2000.pt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)

for key, value in ckpt.items():
    if isinstance(value, dict) and value and all(torch.is_tensor(v) for v in value.values()):
        print(f"\n{key}: state_dict with {len(value)} tensors")
        for name, tensor in value.items():
            print(f"    {name:60s} {tuple(tensor.shape)}")
    elif isinstance(value, dict):
        print(f"\n{key}: dict")
        for name, item in value.items():
            print(f"    {name:30s} {item!r}")
    else:
        print(f"\n{key}: {type(value).__name__} = {value!r}")


epoch: int = 2000

model_state_dict: state_dict with 83 tensors
    start_last_node                                              (128,)
    encoder.embedding_depot.weight                               (128, 2)
    encoder.embedding_depot.bias                                 (128,)
    encoder.embedding_node.weight                                (128, 3)
    encoder.embedding_node.bias                                  (128,)
    encoder.layers.0.Wq.weight                                   (128, 128)
    encoder.layers.0.Wk.weight                                   (128, 128)
    encoder.layers.0.Wv.weight                                   (128, 128)
    encoder.layers.0.multi_head_combine.weight                   (128, 128)
    encoder.layers.0.multi_head_combine.bias                     (128,)
    encoder.layers.0.add_n_normalization_1.norm.weight           (128,)
    encoder.layers.0.add_n_normalization_1.norm.bias             (128,)
    encoder.layers.0.feed_forward.W1.weight        